In [ ]:
# colab workspace setup as my GPU gave up during the most important part of the project !! :DD
# :DDDDDDDDDDDDDDDDD

# driver mount
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
%%bash
# clone specific branch of the project

set -euo pipefail
if [ ! -d /content/local-ijepa/.git ]; then
  git clone -b test/execute-large-runs --depth 1 https://github.com/csanadlajko/local-ijepa.git /content/local-ijepa
fi

In [ ]:
%%bash 
# dowload requirements

cd /content/local-ijepa

# keep torch workspaces torch by not overwriting with requirements.txt
pip install -q huggingface_hub transformers

In [ ]:
%%bash
# download coco dset to colab ws

set -euo pipefail
mkdir -p /content/coco
cd /content/coco
if [ ! -d train2017 ] || [ "$(find train2017 -name '*.jpg' | wc -l)" -lt 100000 ]; then
  wget -c -O train2017.zip http://images.cocodataset.org/zips/train2017.zip
  unzip -q -o train2017.zip
  rm -f train2017.zip
fi
if [ ! -f annotations/instances_train2017.json ]; then
  wget -c -O annotations_trainval2017.zip http://images.cocodataset.org/annotations/annotations_trainval2017.zip
  unzip -q -o annotations_trainval2017.zip
  rm -f annotations_trainval2017.zip
fi
ls -lh annotations/instances_train2017.json

In [ ]:
%%bash
# run

set -euo pipefail
HEAD="euclidean"
EPOCHS=40
RUN="/content/drive/MyDrive/tdk-runs/$HEAD"
STAMP=$(date -u +%Y-%m-%dT%H%M%SZ)
mkdir -p "$RUN"
cd /content/local-ijepa
python3 -u local_jepa.py \
    --dataset coco \
    --coco_train_folder /content/coco/train2017 \
    --coco_train_annotation /content/coco/annotations/instances_train2017.json \
    --result_folder "$RUN" \
    --image_size 256 \
    --patch_size 16 \
    --embed_dim 256 \
    --mlp_dim 512 \
    --depth 6 \
    --num_heads 8 \
    --num_target 4 \
    --batch_size 16 \
    --epochs "$EPOCHS" \
    --lr 0.0001 \
    --finetune_lr 0.003 \
    --num_classes 80 \
    --num_block_categories 81 \
    --multimodal_run n \
    --debug y \
    --geometry "$HEAD" \
    --load_backbone y \
    --bb_run_id "2026-10-30T224556Z" \
    2>&1 | tee "$RUN/train_${STAMP}.log"